In [1]:
import tensorflow as tf
import cv2
import numpy as np
import math

In [2]:
patch_size = 15
kernel_size = patch_size
image_size = 256
descriptor_dimension = 4

max_rotation = 10
max_skew = 0.3

learning_rate = 0.001
optimizer = tf.keras.optimizers.Adam(learning_rate = learning_rate)
batch_size = 256
epochs = 10000
model_path = "filters_model.h5"
backup_model_path = "backup_filters_model.h5"

In [3]:
centre_to_top_right = np.array([[1,0,-image_size//2],
                                [0,1,-image_size//2],
                                [0,0,1]])
top_right_to_centre = np.array([[1,0,image_size//2],
                                [0,1,image_size//2],
                                [0,0,1]])

def create_rotational_matrix(rotation, axis = "z"):
    rotation = np.radians(rotation)
    s = np.sin(rotation)
    c = np.cos(rotation)
    if axis == "z":
        return np.array([[c, -s, 0],
                          [s, c, 0],
                          [0, 0, 1]])
        
def create_skew_matrix(skew_x, skew_y):
     return np.array([[1, skew_x, 0],
                    [skew_y, 1, 0],
                    [0, 0, 1]])

In [4]:
def get_random_homography_matrix():
    rotation = np.random.uniform(0,max_rotation)
    skew_x = np.random.uniform(0,max_skew)
    skew_y = np.random.uniform(0,max_skew)

    rotation_matrix = create_rotational_matrix(rotation)
    skew_matrix = create_skew_matrix(skew_x, skew_y)
    homography_matrix = np.matmul(top_right_to_centre, np.matmul(np.matmul(skew_matrix, rotation_matrix), centre_to_top_right))
    return homography_matrix

In [5]:
class CustomDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, image_path, batch_size = 128):
        self.img = cv2.imread(image_path)
        self.img = cv2.resize(self.img, (image_size, image_size))
        self.img = cv2.cvtColor(self.img, cv2.COLOR_RGB2GRAY)
        self.img = self.img/255
        self.batchsize = batch_size  
                
    def __len__(self):
        return math.ceil(((image_size - patch_size)**2)/self.batchsize)
        
    def __getitem__(self, batch):
        original_images = []
        matches = []
        non_matches = []
        for _ in range(self.batchsize):
            x = np.random.randint(0,image_size - patch_size)
            y = np.random.randint(0,image_size - patch_size)
            current_patch = self.img[x:x+patch_size, y:y+patch_size]
            transformed_current_patch = cv2.warpPerspective(current_patch, get_random_homography_matrix(), (patch_size, patch_size))
            
            current_patch = np.expand_dims(current_patch, axis = -1)
            transformed_current_patch = np.expand_dims(transformed_current_patch, axis = -1)
            
            original_images.append(current_patch)
            matches.append(transformed_current_patch)
            
            X = np.random.randint(0,image_size - patch_size)
            Y = np.random.randint(0,image_size - patch_size)
            
            while(x==X & y==Y):
                X = np.random.randint(0,image_size - patch_size)
                Y = np.random.randint(0,image_size - patch_size)
                
            current_patch = self.img[X:X+patch_size, Y:Y+patch_size]
            transformed_current_patch = cv2.warpPerspective(current_patch, get_random_homography_matrix(), (patch_size, patch_size))
            transformed_current_patch = np.expand_dims(transformed_current_patch, axis = -1)
            non_matches.append(transformed_current_patch)
        
        return np.array(original_images), np.array(matches), np.array(non_matches)

In [6]:
class ChannelNormalization(tf.keras.layers.Layer):
    def __init__(self):
        super(ChannelNormalization, self).__init__()

    def call(self, inputs):
        return tf.keras.backend.l2_normalize(inputs, axis=-1)
    
    def compute_output_shape(self, input_shape):
        return input_shape
    
    def build(self, input_shape):
        super(ChannelNormalization, self).build(input_shape)
        
    def get_config(self):
        return super(ChannelNormalization, self).get_config()
    
    @classmethod
    def from_config(cls, config):
        instance = cls()
        return instance

In [7]:
class ClippingLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(ClippingLayer, self).__init__()

    def call(self, inputs):
        return tf.keras.backend.clip(inputs, min_value=0.0, max_value=256.0)
    
    def compute_output_shape(self, input_shape):
        return input_shape
    
    def build(self, input_shape):
        super(ClippingLayer, self).build(input_shape)
        
    def get_config(self):
        return super(ClippingLayer, self).get_config()
    
    @classmethod
    def from_config(cls, config):
        instance = cls()
        return instance

In [8]:
def get_model(input_shape, descriptor_dimension):
    inp = tf.keras.layers.Input(input_shape)
    out = tf.keras.layers.Conv2D(descriptor_dimension, kernel_size)(inp)
    # out = ClippingLayer()(out)
    out = ChannelNormalization()(out)
    
    model = tf.keras.models.Model(inputs = inp, outputs = out)
    return model

In [9]:
def custom_loss(original_desc, matches, non_matches):
    original_desc = tf.squeeze(original_desc)
    matches = tf.squeeze(matches)
    non_matches = tf.squeeze(non_matches)
    
    return tf.reduce_mean((1 - tf.matmul(original_desc, tf.transpose(matches))) + \
                            abs(tf.matmul(original_desc, tf.transpose(matches))) + \
                            abs(tf.matmul(matches, tf.transpose(non_matches))))
    

In [10]:
model = get_model((patch_size, patch_size, 1), descriptor_dimension)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 15, 15, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 1, 1, 4)        │           904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ channel_normalization           │ (None, 1, 1, 4)        │             0 │
│ (ChannelNormalization)          │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 904 (3.53 KB)

 Trainable params: 904 (3.53 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
data_gen = CustomDataGenerator("test/img/cinema1.jpeg", batch_size)

for epoch in range(epochs):
    epoch_loss = 0
    print(f"Epoch {epoch} starting...")
    for batch in range(data_gen.__len__()):
        all_patches, matches, non_matches = data_gen.__getitem__(batch)
        with tf.GradientTape() as tape:
            all_predictions = model(all_patches)
            matches_predictions = model(matches)
            non_matches_predictions = model(non_matches)
            loss = custom_loss(all_predictions, matches_predictions, non_matches_predictions)
            # print(loss)
            epoch_loss += loss
            gradients = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        
        # if batch%5==0:
        #     print(loss.numpy())
    # val_generator.on_epoch_end()
    
    try:
        model.save(model_path)
        model.save(backup_model_path)
    except Exception as e:
        print("========= Note ================")
        print("cant save last model")
        print(e)
    
    print(epoch_loss.numpy()/data_gen.__len__())
    epoch_loss = 0

Epoch 0 starting...


1.8495287117979076
Epoch 1 starting...


1.839869158908659
Epoch 2 starting...


1.836893224506126
Epoch 3 starting...


1.831645814332668
Epoch 4 starting...


...

1.8162240856019412
Epoch 42 starting...


1.8153534607740225
Epoch 43 starting...


1.8173065857740225
Epoch 44 starting...


1.816604143722467
Epoch 45 starting...


1.8191208356277533
Epoch 46 starting...
